# Testing SST implementation of SHRED to understand

Dataset: Weekly mean sea-surface temperature as given by the NOAA Optimum Interpolation SST V2 dataset [[1]](https://psl.noaa.gov/data/gridded/data.noaa.oisst.v2.html)

Code: Adapted from the example in pyshred [[2]](https://github.com/Jan-Williams/pyshred/blob/main/example.ipynb) (so I understand what happened, the helper functions have been stored directly in this notebook)

In [1]:
import numpy as np

import torch
from scipy.io import loadmat
from sklearn.preprocessing import MinMaxScaler

import models  # https://github.com/Jan-Williams/pyshred/blob/main/models.py 

### Load data

In [2]:
def load_data(path='Data/SST_data.mat'):
    load_X = loadmat(path)['Z'].T
    mean_X = np.mean(load_X, axis=0)
    sst_locs = np.where(mean_X != 0)[0]
    return load_X[:, sst_locs]

In [3]:
data = load_data()

From Mobile-Sensor SHRED paper [[3]](https://ieeexplore.ieee.org/document/10584544):

Weekly mean SST from 1992 - 2019

1400 snapshots of a 180 x 360 grid, with landmass excluded leaves 44,219 spatial locations corresponding to the sea surface

In [4]:
data

array([[-1.79999996, -1.79999996, -1.79999996, ..., -1.30999997,
        -1.29999997, -1.26999997],
       [-1.79999996, -1.79999996, -1.79999996, ..., -1.31999997,
        -1.29999997, -1.26999997],
       [-1.79999996, -1.79999996, -1.79999996, ..., -1.16999997,
        -1.14999997, -1.10999998],
       ...,
       [-1.79999996, -1.79999996, -1.79999996, ..., -1.46999997,
        -1.45999997, -1.44999997],
       [-1.79999996, -1.79999996, -1.79999996, ..., -1.54999997,
        -1.53999997, -1.52999997],
       [-1.79999996, -1.79999996, -1.79999996, ..., -1.61999996,
        -1.60999996, -1.59999996]], shape=(1400, 44219))

In [5]:
n = data.shape[0]  # Number of snapshots
m = data.shape[1]  # Number of spatial locations

In [6]:
len(data)

1400

### Preprocess
Choose 3 sensor locations randomly

In [7]:
num_sensors = 3 
lags = 52  # one year of measurements
sensor_locations = np.random.choice(m, size=num_sensors, replace=False)  # Pick three locations from the m locations in the dataset
sensor_locations

array([32062, 35065, 20271])

Select indices to divide the data into training, validation, and test sets

get 1000 training snapshots, 174 test and 174 validation snapshots. Randomly chosen across entire dataset - not done by time.

In [8]:
train_indices = np.random.choice(n - lags, size=1000, replace=False)
mask = np.ones(n - lags)
mask[train_indices] = 0
valid_test_indices = np.arange(0, n - lags)[np.where(mask!=0)[0]]
valid_indices = valid_test_indices[::2]
test_indices = valid_test_indices[1::2]

For training: use MinMaxScaler() on training data (normalisation step)

In [9]:
sc = MinMaxScaler()
sc = sc.fit(data[train_indices])  # Computes min and max from training data only
transformed_X = sc.transform(data)  # Normalises data

Organize the data such that the inputs are of shape (batch_size, lags, num_sensors) with corresponding outputs of size (batch_size, state_dimension)

In [10]:
# Generate input sequences to a SHRED model
all_data_in = np.zeros((n - lags, lags, num_sensors))
for i in range(len(all_data_in)):
    all_data_in[i] = transformed_X[i:i+lags, sensor_locations]  # stores values between i and 52 weeks, at 3 sensor locations

In [11]:
print(len(all_data_in))  # Number of sequences
print(len(all_data_in[0]))  # sequences of 52 weeks
print(len(all_data_in[0][0]))  # 3 sensor locations

1348
52
3


In [12]:
class TimeSeriesDataset(torch.utils.data.Dataset):
    '''Takes input sequence of sensor measurements with shape (batch size, lags, num_sensors)
    and corresponding measurements of high-dimensional state, return Torch dataset'''
    def __init__(self, X, Y):
        self.X = X
        self.Y = Y
        self.len = X.shape[0]
        
    def __getitem__(self, index):
        return self.X[index], self.Y[index]
    
    def __len__(self):
        return self.len

In [13]:
### Generate training validation and test datasets both for reconstruction of states and forecasting sensors
device = 'cuda' if torch.cuda.is_available() else 'cpu'

train_data_in = torch.tensor(all_data_in[train_indices], dtype=torch.float32).to(device)
valid_data_in = torch.tensor(all_data_in[valid_indices], dtype=torch.float32).to(device)
test_data_in = torch.tensor(all_data_in[test_indices], dtype=torch.float32).to(device)

### -1 to have output be at the same time as final sensor measurements
train_data_out = torch.tensor(transformed_X[train_indices + lags - 1], dtype=torch.float32).to(device)
valid_data_out = torch.tensor(transformed_X[valid_indices + lags - 1], dtype=torch.float32).to(device)
test_data_out = torch.tensor(transformed_X[test_indices + lags - 1], dtype=torch.float32).to(device)

train_dataset = TimeSeriesDataset(train_data_in, train_data_out)
valid_dataset = TimeSeriesDataset(valid_data_in, valid_data_out)
test_dataset = TimeSeriesDataset(test_data_in, test_data_out)

Pass in a sequence of 52 week measurements for 3 sensor locations

In [14]:
print(len(train_data_in))  # Number of sequences
print(len(train_data_in[0]))  # sequences of 52 weeks
print(len(train_data_in[0][0]))  # 3 sensor locations

1000
52
3


Want model to predict entire spatial field from the 52 weeks at 3 sensor locations

To do this, pass it the true data to learn from as labels

In [15]:
print(len(train_data_out))  # Number of sequences
print(len(train_data_out[0]))  # Number of spatial locations

1000
44219


In [16]:
shred = models.SHRED(num_sensors, m, hidden_size=64, hidden_layers=2, l1=350, l2=400, dropout=0.1).to(device)
validation_errors = models.fit(shred, train_dataset, valid_dataset, batch_size=64, num_epochs=1000, lr=1e-3, verbose=True, patience=5)

Training epoch 1
Error tensor(0.4733)
Training epoch 20
Error tensor(0.2312)
Training epoch 40
Error tensor(0.2214)
Training epoch 60
Error tensor(0.2117)
Training epoch 80
Error tensor(0.2085)
Training epoch 100
Error tensor(0.2005)
Training epoch 120
Error tensor(0.1994)
Training epoch 140
Error tensor(0.1872)
Training epoch 160
Error tensor(0.1810)
Training epoch 180
Error tensor(0.1738)
Training epoch 200
Error tensor(0.1712)
Training epoch 220
Error tensor(0.1553)
Training epoch 240
Error tensor(0.1477)
Training epoch 260
Error tensor(0.1409)
Training epoch 280
Error tensor(0.1372)
Training epoch 300
Error tensor(0.1313)
Training epoch 320
Error tensor(0.1301)
Training epoch 340
Error tensor(0.1247)
Training epoch 360
Error tensor(0.1239)
Training epoch 380
Error tensor(0.1218)
Training epoch 400
Error tensor(0.1198)
Training epoch 420
Error tensor(0.1205)
Training epoch 440
Error tensor(0.1180)
Training epoch 460
Error tensor(0.1178)
Training epoch 480
Error tensor(0.1174)
Traini

In [ ]:
test_recons = sc.inverse_transform(shre                                                                                                                              d(test_dataset.X).detach().cpu().numpy())
test_ground_truth = sc.inverse_transform(test_dataset.Y.detach().cpu().numpy())
print('Test Reconstruction Error: ')
print(np.linalg.norm(test_recons - test_ground_truth) / np.linalg.norm(test_ground_truth))

Test Reconstruction Error: 
0.020116856


In [18]:
test_recons

array([[-1.8000191, -1.8000226, -1.8000183, ..., -1.7086729, -1.7205293,
        -1.7301364],
       [-1.8000554, -1.8000551, -1.8000555, ..., -1.7541975, -1.7494798,
        -1.7438635],
       [-1.7999175, -1.7999161, -1.7999175, ..., -1.7558928, -1.7467964,
        -1.7437252],
       ...,
       [-1.7997214, -1.7997214, -1.7997212, ..., -1.5946207, -1.5873512,
        -1.5817542],
       [-1.7995847, -1.7995836, -1.7995849, ..., -1.6547122, -1.6505338,
        -1.6422698],
       [-1.7999033, -1.7999026, -1.7999034, ..., -1.6355301, -1.6247284,
        -1.6126953]], shape=(174, 44219), dtype=float32)

Spatial Coverage [[1]](https://psl.noaa.gov/data/gridded/data.noaa.oisst.v2.html):

1.0 degree latitude x 1.0 degree longitude global grid (180x360).
89.5N - 89.5S, 0.5E - 359.5E.

BUT we don't know how the dataset was made... and hence exactly what coordinates correspond to each point...